# Module 11 project-exercise demo — History-conditioned sampling and quantized layer error

**Implementation demo, not a completed full-scale project.** Demo 1 generates history-dependent sequences and compares the joint law of the first two tokens with an exact calculation. Demo 2 compares value grids on one layer, accounts for scales, and separates weight error from output error. Data are synthetic; neither a trained language model nor a GPU server is involved.
All inputs are small, deterministic, offline, and CPU-friendly. The goal is to
validate interfaces, metrics, invariants, and report structure before spending
GPU or API budget.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"reproducible smoke tier; seed={SEED}")
DEMO_LANGUAGE = 'en'


## Demo 1. A sequence law that depends on history

A three-token Markov model chooses its next probability row from the preceding token, with a separate row for the start. The draft proposes up to three tokens. One simulated target call stores every verification row and the bonus row; rejection resumes from the corrected history. Generate 12,000 length-four sequences and compare first-two-token frequencies with the exact product of the initial and transition probabilities.

In [ ]:
"""M11 minimal probability and grouped-value teaching kernels."""
import numpy as np


def probability_vector(values, *, name="p", atol=1e-12):
    """Validate a distribution without hidden clipping or renormalization."""
    x = np.asarray(values, dtype=np.float64)
    if x.ndim != 1 or x.size < 2 or not np.isfinite(x).all():
        raise ValueError(f"{name}: expected a finite one-dimensional vector")
    if np.any(x < 0) or not np.isclose(x.sum(), 1.0, rtol=0.0, atol=atol):
        raise ValueError(f"{name}: probabilities must be >= 0 and sum to 1")
    return x


def rejection_kernel(p_values, q_values):
    """Acceptance probabilities, residual, and exact one-position balance."""
    p = probability_vector(p_values, name="p")
    q = probability_vector(q_values, name="q")
    if p.shape != q.shape:
        raise ValueError("p and q must use the same vocabulary")

    acceptance = np.zeros_like(p)
    support = q > 0  # q_t=0 means the proposer cannot emit token t
    acceptance[support] = np.minimum(1.0, p[support] / q[support])

    overlap = np.minimum(p, q)
    positive = np.maximum(p - q, 0.0)
    residual_mass = positive.sum()
    residual = (
        positive / residual_mass
        if residual_mass > 0.0
        else np.zeros_like(p)
    )
    balance = overlap + residual_mass * residual
    np.testing.assert_allclose(balance, p, rtol=0.0, atol=1e-12)
    return acceptance, residual, balance


def quantize_grouped_e2m1(values, group_size=16):
    """Teaching E2M1-like grid with one scale per group."""
    grid = np.array(
        [-6, -4, -3, -2, -1.5, -1, -0.5, 0,
          0.5, 1, 1.5, 2, 3, 4, 6],
        dtype=np.float64,
    )
    x = np.asarray(values, dtype=np.float64)
    if x.ndim != 1 or x.size == 0 or not np.isfinite(x).all():
        raise ValueError("expected a non-empty finite one-dimensional vector")
    if isinstance(group_size, bool) or not isinstance(group_size, (int, np.integer)) or group_size <= 0 or x.size % group_size:
        raise ValueError("length must be divisible by a positive group_size")

    groups = x.reshape(-1, group_size)
    max_abs = np.max(np.abs(groups), axis=1, keepdims=True)
    scale = np.where(max_abs > 0, max_abs / 6.0, 1.0)
    normalized = groups / scale
    codes = np.abs(normalized[..., None] - grid).argmin(axis=-1)
    return (grid[codes] * scale).reshape(x.shape)

# Rows 0..2 depend on the preceding token; row 3 is the initial distribution.
P=np.array([[.7,.2,.1],[.1,.7,.2],[.2,.1,.7],[.4,.35,.25]])
Q=np.array([[.55,.25,.2],[.2,.6,.2],[.3,.15,.55],[.3,.4,.3]])
def row(table, history):
    return table[history[-1] if history else 3]

class Target:
    def __init__(self):
        self.calls=0
    def verify(self, prefix, draft):
        # One stored batch result, including the bonus distribution.
        self.calls+=1
        histories=[prefix+draft[:i] for i in range(len(draft)+1)]
        return np.stack([row(P,h) for h in histories])

def generate(rng, length, gamma=3):
    target=Target(); history=[]; proposed=accepted=0
    while len(history)<length:
        draft=[]; q_rows=[]
        for _ in range(min(gamma,length-len(history))):
            q=row(Q,history+draft);q_rows.append(q.copy())
            draft.append(int(rng.choice(3,p=q)))
        proposed+=len(draft)
        p_rows=target.verify(history,draft)
        emitted=[]
        for i, token in enumerate(draft):
            a,residual,_=rejection_kernel(p_rows[i],q_rows[i])
            if rng.random()<a[token]:
                emitted.append(token);accepted+=1
            else:
                emitted.append(int(rng.choice(3,p=residual)))
                break
        else:
            emitted.append(int(rng.choice(3,p=p_rows[-1])))
        # No EOS in this fixed-length experiment. Remove any bonus beyond the limit.
        history.extend(emitted[:length-len(history)])
    return history,target.calls,proposed,accepted

rng=np.random.default_rng(SEED)
records=[generate(rng,4) for _ in range(12000)]
joint=np.zeros((3,3))
for tokens,_,_,_ in records:
    joint[tokens[0],tokens[1]]+=1
joint/=len(records)
exact=P[3,:,None]*P[:3]
tv=.5*np.abs(joint-exact).sum()
assert tv<.025
calls=sum(r[1] for r in records)
assert len(records)<=calls<=4*len(records)
print("joint-law TV:",tv,"target batch calls:",calls,"AR calls:",4*len(records))
print("emitted tokens / target call:",4*len(records)/calls)
print("This is a call-count experiment, not a GPU speed benchmark.")
# With no proposals, each stored target batch supplies one ordinary token.
baseline = generate(np.random.default_rng(SEED), 4, gamma=0)
assert baseline[1:] == (4, 0, 0)
# Each target row uses the history BEFORE its candidate. A shifted row fails.
wrong=P[3,:,None]*P[:3][[1,2,0]]
assert .5*np.abs(wrong-exact).sum()>.2

Small total variation supports agreement with the analytical joint law in this experiment; it supplements rather than replaces the proof. Deliberately shifted target rows produce a different joint distribution, showing why marginal checks alone would be insufficient. A Target call here constructs a table, so fewer calls are not a measured GPU speedup. At gamma=0, four tokens take four calls.

## Demo 2. From weight error to output error

Use a 16×64 weight matrix and 256×64 input matrix. One weight column contains outliers, while one input channel has greater amplitude. Both quantizers use groups of sixteen and full-precision scales, so the experiment compares value grids rather than deployed NVFP4 and INT4 formats. Equal-MSE perturbations on two different channels then isolate the input’s role in output error.

In [ ]:
rng=np.random.default_rng(SEED+1)
w=rng.normal(size=(16,64));w[:,3]*=6
x=rng.normal(size=(256,64));x[:,0]*=8
grid=np.array([-6,-4,-3,-2,-1.5,-1,-.5,0,.5,1,1.5,2,3,4,6])
def quantize_rows(w, fp4):
    groups=w.reshape(w.shape[0],-1,16)
    peak=np.abs(groups).max(-1,keepdims=True)
    scale=np.where(peak>0,peak/(6 if fp4 else 7),1.)
    normalized=groups/scale
    code=grid[np.abs(normalized[...,None]-grid).argmin(-1)] if fp4 else np.rint(normalized).clip(-7,7)
    return (code*scale).reshape(w.shape),scale.size

y=x@w.T
for fp4 in (False,True):
    restored,n_scales=quantize_rows(w,fp4)
    weight_mse=np.mean((w-restored)**2)
    output_error=np.linalg.norm(x@restored.T-y)/np.linalg.norm(y)
    ideal_bytes=(w.size+1)//2+n_scales*4
    print("E2M1" if fp4 else "INT4","weight MSE:",weight_mse,
          "relative output error:",output_error,"ideal payload + float32 scales B:",ideal_bytes)
    assert np.isfinite(output_error)
assert np.array_equal(quantize_rows(np.zeros_like(w),True)[0],np.zeros_like(w))
# Equal weight MSE can affect different input channels very differently.
e0=np.zeros_like(w);e1=np.zeros_like(w);e0[:,0]=.1;e1[:,1]=.1
assert np.isclose(np.mean(e0**2),np.mean(e1**2))
out0=np.mean((x@e0.T)**2);out1=np.mean((x@e1.T)**2)
assert out0>20*out1
print("equal weight MSE; output-MSE ratio:",out0/out1)
# Decoded arrays are float64; the byte count assumes packing that this code does not implement.

Weight MSE depends only on the original and reconstructed matrices; output error also depends on the input distribution. Equal perturbations in different channels therefore have different effects, and a value-grid ranking can change with the workload. The byte estimate assumes packed codes and scales; the experiment actually retains float64 arrays.

## Scaling to the full project

First vary the draft distribution, gamma, and input-channel activity. An optional extension replaces the tables with a local model and engine, preserving alignment, rollback, and sampling checks. Measure latency and goodput at three loads. Quantization also needs held-out quality evaluation and measured artifact size; layer MSE is not a model benchmark.

### Minimum report

1. Data, model, and code revisions, plus the random seed.
2. The primary metric, plus quality, cost, latency, and memory where applicable.
3. A baseline, a matched budget, and at least one negative control.
4. Failed runs and the limits of the supported conclusions.